# Gop TCGA-LUAD + TCGA-LUSC thanh 1 file `tcga_expr.h5`

`tcga_download.ipynb` luu **rieng 1 file H5 cho moi project**
(`tcga_luad_expr.h5`, `tcga_lusc_expr.h5`, moi file co 3 key bare: `counts`,
`tpm`, `pheno`). Cac notebook `fair-comparison/*.ipynb` (`DATA_SOURCE="tcga"`
hoac `"merged"`) lai doc thang 1 file duy nhat `tcga_expr.h5` voi key co tien
to project (`TCGA-LUAD/tpm`, `TCGA-LUAD/pheno`, `TCGA-LUSC/tpm`,
`TCGA-LUSC/pheno`, ...) roi **tu gop 2 project lai** khi tinh X/y (xem ham
`_load_tcga_direct` trong cac notebook do).

Notebook nay lam đúng buoc trung gian con thieu: doc 2 file rieng do
`tcga_download.ipynb` tao ra, roi ghi lai thanh 1 file `tcga_expr.h5` voi
schema (key co tien to project) ma `_load_tcga_direct`/`_load_tcga_gene_level`
mong doi -- sau buoc nay, chi can dat `DATA_SOURCE = "tcga"` trong bat ky
notebook `fair-comparison/*.ipynb` nao la chay duoc tren TCGA-LUAD+LUSC gop
chung (khong sua gi them ma no ca).

Ly do gop LUAD+LUSC thay vi chay rieng: bai toan o day la phan loai **giai
doan** (stage_encoded, dung chung giua LUAD/LUSC), khong phai phan biet
subtype, va ca 2 project cung dung chung 1 pipeline STAR-Counts/GENCODE cua
GDC nen khong can hieu chinh batch (ComBat) nhu khi gop GEO+TCGA.


## 0. Thiet lap

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd


def resolve_dataset_dir(dirname: str, required_files) -> str:
    """
    Tim thu muc chua du lieu da tai boi tcga_download.ipynb. Uu tien Google
    Drive (Colab), fallback ve cac duong dan local pho bien.
    """
    candidates = []

    env_dir = os.environ.get("KLTN_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir) / dirname)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / dirname,
        cwd.parent / "data" / dirname,
        Path("/content/drive/MyDrive/KLTN/data") / dirname,
        Path(r"D:\KLTN\data") / dirname,
    ])

    for c in candidates:
        if all((c / f).exists() for f in required_files):
            return str(c)

    checked = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(
        f"Khong tim thay {required_files} trong bat ky thu muc nao:\n{checked}\n\n"
        "Hay chay tcga_download.ipynb truoc (can ca 2 file tcga_luad_expr.h5 "
        "va tcga_lusc_expr.h5), hoac dat bien moi truong KLTN_DATA_DIR tro toi "
        "thu muc cha chua 'tcga/'."
    )


PROJECTS = ["TCGA-LUAD", "TCGA-LUSC"]
SRC_FILES = {
    "TCGA-LUAD": "tcga_luad_expr.h5",
    "TCGA-LUSC": "tcga_lusc_expr.h5",
}

TCGA_DIR = resolve_dataset_dir("tcga", list(SRC_FILES.values()))
OUT_H5_PATH = os.path.join(TCGA_DIR, "tcga_expr.h5")

print("TCGA_DIR    :", TCGA_DIR)
print("OUT_H5_PATH :", OUT_H5_PATH)


## 1. Doc 2 file rieng (moi file: key bare `counts`/`tpm`/`pheno`)

In [ ]:
KEYS = ["counts", "tpm", "pheno"]

raw = {}  # raw[project][key] -> DataFrame
for p in PROJECTS:
    path = os.path.join(TCGA_DIR, SRC_FILES[p])
    raw[p] = {k: pd.read_hdf(path, key=k) for k in KEYS}
    print(f"{p}: counts {raw[p]['counts'].shape}, tpm {raw[p]['tpm'].shape}, "
          f"pheno {raw[p]['pheno'].shape}")


## 2. Kiem tra nhanh truoc khi gop

`_load_tcga_direct` gop 2 project bang `pd.concat([...], axis=1)` theo INDEX
(gene symbol) -- neu 2 project lech gene symbol nhieu, cac gene thieu se
thanh NaN va lam hong buoc train phia sau. In ty le trung nhau de kiem tra;
neu thap bat thuong thi dien LUAD/LUSC co the da duoc xu ly khac nhau va can
xem lai `tcga_download.ipynb`.


In [ ]:
genes_luad = set(raw["TCGA-LUAD"]["tpm"].index)
genes_lusc = set(raw["TCGA-LUSC"]["tpm"].index)
common = genes_luad & genes_lusc

print(f"So gene (symbol) TCGA-LUAD : {len(genes_luad)}")
print(f"So gene (symbol) TCGA-LUSC : {len(genes_lusc)}")
print(f"So gene chung              : {len(common)} "
      f"({100 * len(common) / max(len(genes_luad), len(genes_lusc)):.1f}%)")

assert len(common) / max(len(genes_luad), len(genes_lusc)) > 0.95, (
    "Ty le gene chung giua LUAD/LUSC qua thap -- kiem tra lai truoc khi gop."
)
print("OK: ty le gene chung du cao de gop truc tiep (khong can align gene).")


## 3. Ghi `tcga_expr.h5` (key co tien to project)

Dung dung schema ma `_load_tcga_direct`/`_load_tcga_gene_level` trong
`fair-comparison/*.ipynb` mong doi: `{project}/counts`, `{project}/tpm`,
`{project}/pheno`.


In [ ]:
with pd.HDFStore(OUT_H5_PATH, mode="w") as store:
    for p in PROJECTS:
        for k in KEYS:
            store.put(f"{p}/{k}", raw[p][k])

print(f"Da ghi: {OUT_H5_PATH}")
for p in PROJECTS:
    print(f"  {p}/counts : {raw[p]['counts'].shape}")
    print(f"  {p}/tpm    : {raw[p]['tpm'].shape}")
    print(f"  {p}/pheno  : {raw[p]['pheno'].shape}")


## 4. Kiem tra lai bang chinh ham `_load_tcga_direct`

Copy y het ham dung trong `fair-comparison/*.ipynb` de chac chan file vua ghi
doc duoc dung nhu cac notebook do se doc.


In [ ]:
def _load_tcga_direct(tcga_dir: str, projects=("TCGA-LUAD", "TCGA-LUSC"), value: str = "tpm"):
    h5_path = os.path.join(tcga_dir, "tcga_expr.h5")

    frames, ys = [], []
    for p in projects:
        expr = pd.read_hdf(h5_path, key=f"{p}/{value}")
        pheno = pd.read_hdf(h5_path, key=f"{p}/pheno")
        y_p = pheno.loc[pheno.index.intersection(expr.columns), "stage_encoded"]
        cols = [c for c in expr.columns if c in y_p.index]
        e = expr[cols]
        if value == "tpm":
            e = np.log2(e + 1)
        frames.append(e)
        ys.append(y_p.loc[cols])

    gene_expr = pd.concat(frames, axis=1)
    y = pd.concat(ys)
    X = gene_expr.T
    X.index.name = "sample_id"
    return X, y.values


STAGE_TO_CODE = {"Normal": 0, "I": 1, "II": 2, "III": 3, "IV": 4}
CODE_TO_STAGE = {v: k for k, v in STAGE_TO_CODE.items()}

X, y = _load_tcga_direct(TCGA_DIR)
print("X shape:", X.shape)
print("So gia tri NaN trong X:", int(np.isnan(X.values).sum()))
print("Phan bo nhan:", {CODE_TO_STAGE.get(k, k): v
                        for k, v in pd.Series(y).value_counts().sort_index().to_dict().items()})


In [ ]:
# Luu lai chinh notebook nay (bao gom output) vao Google Drive sau khi chay xong.
try:
    from google.colab import _message
    _message.blocking_request("save_notebook")
    print("Da yeu cau Colab luu notebook nay vao Google Drive.")
except Exception as e:
    print("Khong the tu luu notebook (co le dang chay ngoai Colab, "
          "hoac notebook chua duoc mo tu Drive):", e)
